In [0]:
df = spark.read.parquet("/Volumes/workspace/default/taxi_data/yellow_tripdata_2025*.parquet")
df.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-05-01 00:07:06|  2025-05-01 00:24:15|              1|          3.7|         1|                 N|         140|    

In [0]:
df.printSchema()
df.show(5)
df.count()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

48722602

In [0]:
df = df.select(
    "tpep_pickup_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "PULocationID"
)


In [0]:
from pyspark.sql.functions import col

df = df.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0)
)


In [0]:
from pyspark.sql.functions import hour, dayofweek, month, to_date

df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
df = df.withColumn("pickup_day", dayofweek("tpep_pickup_datetime"))
df = df.withColumn("pickup_month", month("tpep_pickup_datetime"))
df = df.withColumn("pickup_date", to_date("tpep_pickup_datetime"))


In [0]:
df.select("pickup_hour", "pickup_day", "pickup_month").show(5)


+-----------+----------+------------+
|pickup_hour|pickup_day|pickup_month|
+-----------+----------+------------+
|          0|         5|           5|
|          0|         5|           5|
|          0|         5|           5|
|          0|         5|           5|
|          0|         5|           5|
+-----------+----------+------------+
only showing top 5 rows


In [0]:
demand_df = df.groupBy("pickup_date", "pickup_hour") \
    .count() \
    .withColumnRenamed("count", "trip_count")


In [0]:
demand_df.show(5)


+-----------+-----------+----------+
|pickup_date|pickup_hour|trip_count|
+-----------+-----------+----------+
| 2025-05-02|          7|      2983|
| 2025-05-26|         17|      4827|
| 2025-05-11|          7|      1194|
| 2025-05-06|         20|      6392|
| 2025-05-07|          5|       568|
+-----------+-----------+----------+
only showing top 5 rows


In [0]:
demand_df.describe("trip_count").show()


+-------+-----------------+
|summary|       trip_count|
+-------+-----------------+
|  count|             8768|
|   mean|4060.770415145985|
| stddev|2397.636741074835|
|    min|                1|
|    max|             9961|
+-------+-----------------+



In [0]:
from pyspark.sql.functions import percentile_approx

threshold = demand_df.select(
    percentile_approx("trip_count", 0.75)
).collect()[0][0]

print("Threshold:", threshold)


Threshold: 5956


In [0]:
weather_df = spark.read.json(
    "/Volumes/workspace/default/taxi_data/ny_weather_2025.json"
)


In [0]:
weather_df.printSchema()


root
 |-- daily: struct (nullable = true)
 |    |-- precipitation_sum: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_min: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- daily_units: struct (nullable = true)
 |    |-- precipitation_sum: string (nullable = true)
 |    |-- temperature_2m_max: string (nullable = true)
 |    |-- temperature_2m_min: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (nu

In [0]:
from pyspark.sql.functions import explode, arrays_zip, col

weather_flat = weather_df.select(
    explode(
        arrays_zip(
            col("daily.time"),
            col("daily.temperature_2m_max"),
            col("daily.precipitation_sum")
        )
    ).alias("data")
)


In [0]:
weather_flat = weather_flat.select(
    col("data.time").alias("pickup_date"),
    col("data.temperature_2m_max").alias("temperature"),
    col("data.precipitation_sum").alias("precipitation")
)


In [0]:
from pyspark.sql.functions import to_date

weather_flat = weather_flat.withColumn(
    "pickup_date",
    to_date("pickup_date")
)


In [0]:
from pyspark.sql.functions import when

weather_flat = weather_flat.withColumn(
    "rain_flag",
    when(col("precipitation") > 0, 1).otherwise(0)
)

weather_flat.show(5)


+-----------+-----------+-------------+---------+
|pickup_date|temperature|precipitation|rain_flag|
+-----------+-----------+-------------+---------+
| 2025-01-01|       10.9|          1.1|        1|
| 2025-01-02|        5.4|          0.0|        0|
| 2025-01-03|        3.2|          0.0|        0|
| 2025-01-04|       -0.1|          0.0|        0|
| 2025-01-05|        0.3|          0.0|        0|
+-----------+-----------+-------------+---------+
only showing top 5 rows


In [0]:
final_df = demand_df.join(weather_flat, on="pickup_date", how="left")


In [0]:
events_df = spark.read.csv(
    "/Volumes/workspace/default/taxi_data/ny_film_permits.csv",
    header=True,
    inferSchema=True
)
events_df.show(5)


+-------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+---------+----------------+----------------+----------+---------------+--------------------+--------------------+
|eventid|           eventtype|      startdatetime|        enddatetime|          enteredon|         eventagency|         parkingheld|  borough|communityboard_s|policeprecinct_s|  category|subcategoryname|             country|           zipcode_s|
+-------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+---------+----------------+----------------+----------+---------------+--------------------+--------------------+
| 903977|     Shooting Permit|2025-12-06 12:00:00|2025-12-06 08:00:00|2025-12-04 13:21:56|Mayor's Office of...|WEST   48 STREET ...|Manhattan|              5,|             18,|Television|           News|United States of ...|10020, 10036, 10105,|
| 903870|Theater

In [0]:
from pyspark.sql.functions import to_date

events_df = events_df.withColumn(
    "event_date",
    to_date("startdatetime")
)
events_df.select("startdatetime", "event_date").show(5)


+-------------------+----------+
|      startdatetime|event_date|
+-------------------+----------+
|2025-12-06 12:00:00|2025-12-06|
|2025-12-06 12:01:00|2025-12-06|
|2025-12-06 12:01:00|2025-12-06|
|2025-12-05 12:01:00|2025-12-05|
|2025-12-05 13:00:00|2025-12-05|
+-------------------+----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import lit

event_dates = events_df.select("event_date").distinct()
event_dates = event_dates.withColumn("event_flag", lit(1))


In [0]:
final_df = final_df.join(
    event_dates,
    final_df.pickup_date == event_dates.event_date,
    "left"
)
final_df = final_df.fillna({"event_flag": 0})
final_df = final_df.drop("event_date")


In [0]:
from pyspark.sql.functions import when, dayofweek

final_df = final_df.withColumn(
    "is_weekend",
    when(dayofweek("pickup_date").isin([1, 7]), 1).otherwise(0)
)

final_df = final_df.withColumn(
    "peak_hour",
    when(col("pickup_hour").isin([8, 9, 17, 18, 19]), 1).otherwise(0)
)


In [0]:
# Fill nulls FIRST so surge_label has no null inputs
final_df = final_df.fillna({
    "pickup_hour": 0,
    "trip_count": 0,
    "rain_flag": 0,
    "event_flag": 0,
    "is_weekend": 0,
    "peak_hour": 0
})

# Verify no nulls remain
from pyspark.sql.functions import col
final_df.select([
    col(c).isNull().alias(c)
    for c in ["pickup_hour", "trip_count", "rain_flag", "event_flag", "is_weekend", "peak_hour"]
]).show()


+-----------+----------+---------+----------+----------+---------+
|pickup_hour|trip_count|rain_flag|event_flag|is_weekend|peak_hour|
+-----------+----------+---------+----------+----------+---------+
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    false|
|      false|     false|    false|     false|     false|    fa

In [0]:
# Composite surge: high trip count AND at least one contextual factor
# Created AFTER fillna so no null label values
final_df = final_df.withColumn(
    "surge_label",
    (
        (col("trip_count") > threshold) &
        (
            (col("rain_flag") == 1) |
            (col("event_flag") == 1) |
            (col("peak_hour") == 1)
        )
    ).cast("int")
)

# Check label balance — want roughly 20-50% surge=1
final_df.groupBy("surge_label").count().show()


+-----------+-----+
|surge_label|count|
+-----------+-----+
|          0| 7277|
|          1| 1491|
+-----------+-----+



In [0]:
final_df.select(
    "pickup_date", "pickup_hour", "trip_count",
    "rain_flag", "event_flag", "is_weekend", "peak_hour", "surge_label"
).show(10)


+-----------+-----------+----------+---------+----------+----------+---------+-----------+
|pickup_date|pickup_hour|trip_count|rain_flag|event_flag|is_weekend|peak_hour|surge_label|
+-----------+-----------+----------+---------+----------+----------+---------+-----------+
| 2025-05-02|          7|      2983|        0|         0|         0|        0|          0|
| 2025-05-26|         17|      4827|        0|         0|         0|        1|          0|
| 2025-05-11|          7|      1194|        0|         0|         1|        0|          0|
| 2025-05-06|         20|      6392|        1|         0|         0|        0|          1|
| 2025-05-07|          5|       568|        1|         0|         0|        0|          0|
| 2025-05-12|         16|      6660|        0|         0|         0|        0|          0|
| 2025-05-16|          5|       794|        1|         0|         0|        0|          0|
| 2025-05-06|          9|      5246|        1|         0|         0|        1|          0|

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["pickup_hour", "trip_count", "rain_flag", "event_flag", "is_weekend", "peak_hour"],
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)
print("Train:", train_data.count(), "Test:", test_data.count())


Train: 6991 Test: 1777


In [0]:
lr = LogisticRegression(featuresCol="features", labelCol="surge_label")
lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
lr_model = lr_pipeline.fit(train_data)
lr_pred = lr_model.transform(test_data)


In [0]:
from pyspark.ml.feature import StandardScaler as ScalerDT

scaler_dt = ScalerDT(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="surge_label", maxDepth=5)
dt_pipeline = Pipeline(stages=[assembler, scaler_dt, dt])
dt_model = dt_pipeline.fit(train_data)
dt_pred = dt_model.transform(test_data)


In [0]:
from pyspark.ml.feature import StandardScaler as ScalerRF

scaler_rf = ScalerRF(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
rf = RandomForestClassifier(featuresCol="features", labelCol="surge_label", numTrees=50)
rf_pipeline = Pipeline(stages=[assembler, scaler_rf, rf])
rf_model = rf_pipeline.fit(train_data)
rf_pred = rf_model.transform(test_data)


In [0]:
scaler_stage = lr_model.stages[1]
lr_stage     = lr_model.stages[2]

print("MEAN:        ", scaler_stage.mean)
print("STD:         ", scaler_stage.std)
print("COEFFICIENTS:", lr_stage.coefficients.toArray())
print("INTERCEPT:   ", lr_stage.intercept)


MEAN:         [11.500643684737508,4070.9397797167844,0.2457445286797311,0.1604920612215706,0.28293520240308967,0.20898297811471891]
STD:          [6.924565413543322,2394.2810706855425,0.43055855856113967,0.36708804775995013,0.4504574323682295,0.4066112914597255]
COEFFICIENTS: [ 4.27064689e-01  1.17159499e+01  2.58793399e+00  2.24406616e+00
 -4.77018280e-03  1.65122850e+00]
INTERCEPT:    -12.994448563632536


In [0]:
print("LR Confusion Matrix")
lr_pred.groupBy("surge_label", "prediction").count().show()

print("DT Confusion Matrix")
dt_pred.groupBy("surge_label", "prediction").count().show()

print("RF Confusion Matrix")
rf_pred.groupBy("surge_label", "prediction").count().show()


LR Confusion Matrix
+-----------+----------+-----+
|surge_label|prediction|count|
+-----------+----------+-----+
|          0|       0.0| 1448|
|          1|       1.0|  274|
|          1|       0.0|   24|
|          0|       1.0|   31|
+-----------+----------+-----+

DT Confusion Matrix
+-----------+----------+-----+
|surge_label|prediction|count|
+-----------+----------+-----+
|          0|       0.0| 1479|
|          1|       1.0|  298|
+-----------+----------+-----+

RF Confusion Matrix
+-----------+----------+-----+
|surge_label|prediction|count|
+-----------+----------+-----+
|          0|       0.0| 1479|
|          1|       1.0|  298|
+-----------+----------+-----+



In [0]:
from pyspark.sql.functions import col

def get_metrics(pred_df, name):
    tp = pred_df.filter((col("surge_label") == 1) & (col("prediction") == 1)).count()
    tn = pred_df.filter((col("surge_label") == 0) & (col("prediction") == 0)).count()
    fp = pred_df.filter((col("surge_label") == 0) & (col("prediction") == 1)).count()
    fn = pred_df.filter((col("surge_label") == 1) & (col("prediction") == 0)).count()

    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

    return [name, accuracy, precision, recall, f1, tp, tn, fp, fn]

results = [
    get_metrics(lr_pred, "Logistic Regression"),
    get_metrics(dt_pred, "Decision Tree"),
    get_metrics(rf_pred, "Random Forest"),
]

columns = ["Model", "Accuracy", "Precision", "Recall", "F1 Score", "TP", "TN", "FP", "FN"]
results_df = spark.createDataFrame(results, columns)
results_df.show(truncate=False)


+-------------------+------------------+-----------------+------------------+------------------+---+----+---+---+
|Model              |Accuracy          |Precision        |Recall            |F1 Score          |TP |TN  |FP |FN |
+-------------------+------------------+-----------------+------------------+------------------+---+----+---+---+
|Logistic Regression|0.9690489589195272|0.898360655737705|0.9194630872483222|0.9087893864013267|274|1448|31 |24 |
|Decision Tree      |1.0               |1.0              |1.0               |1.0               |298|1479|0  |0  |
|Random Forest      |1.0               |1.0              |1.0               |1.0               |298|1479|0  |0  |
+-------------------+------------------+-----------------+------------------+------------------+---+----+---+---+



In [0]:
for row in results_df.collect():
    print(f"{row['Model']}: Accuracy={row['Accuracy']:.6f}, F1={row['F1 Score']:.6f}")


Logistic Regression: Accuracy=0.969049, F1=0.908789
Decision Tree: Accuracy=1.000000, F1=1.000000
Random Forest: Accuracy=1.000000, F1=1.000000


In [0]:
print(dt_model.stages[2].toDebugString)


DecisionTreeClassificationModel: uid=DecisionTreeClassifier_1f071f3c4c6c, depth=4, numNodes=9, numClasses=2, numFeatures=6
  If (feature 1 <= 0.7900326504864421)
   Predict: 0.0
  Else (feature 1 > 0.7900326504864421)
   If (feature 5 <= 0.7157130851938136)
    If (feature 2 <= 0.590524717868695)
     If (feature 3 <= 0.9248678643997796)
      Predict: 0.0
     Else (feature 3 > 0.9248678643997796)
      Predict: 1.0
    Else (feature 2 > 0.590524717868695)
     Predict: 1.0
   Else (feature 5 > 0.7157130851938136)
    Predict: 1.0



In [0]:
lr_model.write().overwrite().save("/Volumes/workspace/default/taxi_data/models/surge_lr")
print("Model saved.")


Model saved.


In [0]:
pdf = final_df.toPandas()
pdf.to_csv("final_data.csv", index=False)
pdf


,pickup_date,pickup_hour,trip_count,temperature,precipitation,rain_flag,event_flag,is_weekend,peak_hour,surge_label
0,2025-05-02,7,2983,30.1,0.0,0,0,0,0,0
1,2025-05-26,17,4827,22.2,0.0,0,0,0,1,0
2,2025-05-11,7,1194,23.4,0.0,0,0,1,0,0
3,2025-05-06,20,6392,19.1,20.6,1,0,0,0,1
4,2025-05-07,5,568,22.5,0.3,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8763,2025-01-21,10,4991,-8.4,0.4,1,0,0,0,0
8764,2025-01-16,2,342,-0.6,0.3,1,0,0,0,0
8765,2025-01-11,9,2870,0.4,2.0,1,0,1,1,0
8766,2025-01-04,22,5011,-0.1,0.0,0,0,1,0,0
